In [ ]:
#Deps (judge needs internet on to download Qwen2.5-7B-Instruct)
!pip -q install -U "transformers>=4.44" "trl>=0.9" peft bitsandbytes datasets accelerate
!pip -q uninstall -y torchao   # Kaggle ships an old torchao that recent peft rejects; we use bitsandbytes
print("deps installed")

In [ ]:
#Config + helpers. Open-ended task: draft a support reply, scored by an LLM judge (not macro-F1).
import os, json, re, random, statistics
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
TASK   = "/kaggle/working/data"                                  # task data, built next cell
PROBES = "/kaggle/input/datasets/ianrusseladem/pipeline2-data"   # reuse regression probes + rehearsal
OUT    = "/kaggle/working"

CFG = {
    "name": "support-reply-drafting",
    "base_model": "Qwen/Qwen2.5-0.5B-Instruct",   # the model being fine-tuned (cheap); swap freely
    "judge": {"model": "Qwen/Qwen2.5-7B-Instruct", "batch": 4},   # stronger model that scores the answers
    "lora": {"rank": 16, "alpha": 32, "dropout": 0.05, "target_modules": "all-linear"},
    "train": {"epochs": 3, "lr": 2e-4, "max_len": 1024, "batch": 1, "grad_accum": 16,
              "seeds": [0, 1]},   # gate decides on the MEDIAN win-rate across seeds
    "guardrails": {"replay_mix": True},
    # generative gate: accept if the judge prefers the fine-tuned answers on >= min_win_rate of
    # held-out tickets AND no regression probe drops more than max_regression_drop.
    "acceptance": {"min_win_rate": 0.55, "max_regression_drop": 0.05},
    "adjust_on_fail": {"force_replay": True, "lower_lr_factor": 0.5, "reduce_epochs_to": 2},
}

def read_jsonl(p): return [json.loads(l) for l in open(p) if l.strip()]
def score_probes(raw, probes, mode):
    ok = 0
    for o, p in zip(raw, probes):
        want = [a.lower() for a in p["answers"]]; low = o.lower()
        ok += (all(a in low for a in want) if mode=="all" else any(a in low for a in want))
    return ok/len(probes) if probes else 0.0
print("config ready")

In [ ]:
#Build the reply-drafting dataset from Tobi-Bueck (ticket -> agent reply)
from datasets import load_dataset

TEXT_FIELDS=["subject","body"]; ANSWER_CANDS=["answer","response","agent_answer","reply"]
LANG_FIELD,KEEP_LANG="language","en"; N_GOLD,N_TRAIN=60,400; MIN_ANS=40; REPLAY=0.25
INSTR=("You are a customer-support agent. Read the ticket and write a single helpful, professional "
       "reply that resolves it. Reply with the message only.")
def norm(s): return " ".join(str(s).lower().split())
def ticket_text(r): return "\n".join(str(r.get(f,"")).strip() for f in TEXT_FIELDS if r.get(f))
def msgs(t, a=None):
    m=[{"role":"system","content":INSTR},{"role":"user","content":t}]
    if a is not None: m.append({"role":"assistant","content":a})
    return m

random.seed(0); os.makedirs(TASK, exist_ok=True)
ds=load_dataset("Tobi-Bueck/customer-support-tickets", split="train"); cols=ds.column_names
print("columns:", cols)
ans_field=next((f for f in ANSWER_CANDS if f in cols), None)
assert ans_field, f"no answer field in {cols}; set ANSWER_CANDS"
print("answer field:", ans_field)
rows, seen=[], set()
for r in ds:
    if LANG_FIELD in cols and norm(r.get(LANG_FIELD)) not in ("", KEEP_LANG, "english"): continue
    t, a=ticket_text(r), str(r.get(ans_field,"")).strip()
    if not t or len(a)<MIN_ANS: continue
    k=norm(t)
    if k in seen: continue
    seen.add(k); rows.append((t,a))
random.shuffle(rows)
print("usable (ticket, reply) pairs:", len(rows))
assert len(rows)>=N_GOLD+50, "not enough pairs; lower N_GOLD/N_TRAIN"
gold, train=rows[:N_GOLD], rows[N_GOLD:N_GOLD+N_TRAIN]
with open(f"{TASK}/gold.jsonl","w") as f:
    for t,a in gold: f.write(json.dumps({"messages":msgs(t),"reference":a})+"\n")
with open(f"{TASK}/train_synth.jsonl","w") as f:
    for t,a in train: f.write(json.dumps({"messages":msgs(t,a)})+"\n")
mix=[msgs(t,a) for t,a in train]; reh=[]
pool_path=f"{PROBES}/train_synth.jsonl"
if os.path.exists(pool_path):
    pool=[json.loads(x) for x in open(pool_path) if x.strip()]; random.shuffle(pool)
    reh=[p["messages"] for p in pool[:int(len(mix)*REPLAY)] if "messages" in p]
    print(f"replay: added {len(reh)} rehearsal rows")
else: print("replay: no rehearsal pool; train_mix == train_synth")
mixed=mix+reh; random.shuffle(mixed)
with open(f"{TASK}/train_mix.jsonl","w") as f:
    for m in mixed: f.write(json.dumps({"messages":m})+"\n")
print(f"wrote gold={len(gold)} train_synth={len(train)} train_mix={len(mixed)}")

In [ ]:
#Config-driven training (SFT: ticket -> reply)
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

def train_one(data_file, run_name, lr=None, epochs=None, seed=0):
    t, lo = CFG["train"], CFG["lora"]
    lr = t["lr"] if lr is None else lr; epochs = t["epochs"] if epochs is None else epochs
    tok = AutoTokenizer.from_pretrained(CFG["base_model"])
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
    model = AutoModelForCausalLM.from_pretrained(CFG["base_model"], quantization_config=quant,
        torch_dtype=torch.bfloat16, device_map="auto"); model.config.use_cache = False
    lora = LoraConfig(r=lo["rank"], lora_alpha=lo["alpha"], lora_dropout=lo["dropout"],
        bias="none", task_type="CAUSAL_LM", target_modules=lo["target_modules"])
    ds = load_dataset("json", data_files=f"{TASK}/{data_file}", split="train")
    out = f"{OUT}/{run_name}"
    cfg = SFTConfig(output_dir=out, num_train_epochs=epochs, per_device_train_batch_size=t["batch"],
        gradient_accumulation_steps=t["grad_accum"], learning_rate=lr, lr_scheduler_type="cosine",
        warmup_ratio=0.05, logging_steps=10, save_strategy="no", bf16=True,
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        max_length=t["max_len"], packing=False, assistant_only_loss=True, seed=seed, report_to="none")
    tr = SFTTrainer(model=model, args=cfg, train_dataset=ds, peft_config=lora, processing_class=tok)
    r = tr.train(); tr.save_model(out); tok.save_pretrained(out)
    del model, tr; torch.cuda.empty_cache()
    print(f"[train] {run_name}: loss={r.training_loss:.4f} (data={data_file}, lr={lr}, seed={seed}, epochs={epochs})")
    return out

In [ ]:
#Generative eval: produce each model's answers on the held-out tickets + score regression probes
def _load(adapter=None):
    tok = AutoTokenizer.from_pretrained(CFG["base_model"]); tok.padding_side = "left"
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    m = AutoModelForCausalLM.from_pretrained(CFG["base_model"], torch_dtype=torch.bfloat16)
    if adapter:
        from peft import PeftModel; m = PeftModel.from_pretrained(m, adapter)
    return m.to("cuda").eval(), tok

@torch.no_grad()
def _gen(m, tok, prompts, max_new, batch=8):
    outs = []
    for i in range(0, len(prompts), batch):
        ch = prompts[i:i+batch]
        txt = [tok.apply_chat_template(x, add_generation_prompt=True, tokenize=False) for x in ch]
        enc = tok(txt, return_tensors="pt", padding=True, add_special_tokens=False).to(m.device)
        g = m.generate(**enc, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.pad_token_id)
        for j in range(len(ch)):
            outs.append(tok.decode(g[j][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip())
    return outs

def _probe(m, tok, fname, mode, max_new):
    probes = read_jsonl(f"{PROBES}/{fname}")
    raw = _gen(m, tok, [[{"role":"user","content":p["question"]}] for p in probes], max_new)
    return {"n": len(probes), "score": score_probes(raw, probes, mode)}

def evaluate_gen(name, adapter=None):
    gold = read_jsonl(f"{TASK}/gold.jsonl")
    m, tok = _load(adapter)
    answers = _gen(m, tok, [r["messages"] for r in gold], 384)
    res = {"name": name, "tickets": [r["messages"][-1]["content"] for r in gold], "answers": answers,
           "sentinel": _probe(m, tok, "sentinel.jsonl", "any", 128),
           "reasoning": _probe(m, tok, "reasoning_probes.jsonl", "any", 256),
           "tools": _probe(m, tok, "tool_probes.jsonl", "all", 128)}
    json.dump(res, open(f"{OUT}/result_{name}.json","w"), indent=2)
    print(f"[eval-gen] {name}: {len(answers)} answers | sentinel={res['sentinel']['score']:.3f} "
          f"reasoning={res['reasoning']['score']:.3f} tools={res['tools']['score']:.3f}")
    del m; torch.cuda.empty_cache(); return res

In [ ]:
#LLM judge: a stronger model says which reply is better -> win-rate (judged in both A/B orders)
SYSTEM=("You are an impartial judge of customer-support replies. You are given the customer's ticket "
        "and two candidate replies, A and B. Decide which better resolves the ticket: more correct, "
        "helpful, and professional. Do not favour length or position. Briefly justify, then end with "
        "exactly one line: 'VERDICT: A', 'VERDICT: B', or 'VERDICT: tie'.")
USER="Ticket:\n{t}\n\nReply A:\n{a}\n\nReply B:\n{b}\n\nWhich reply is better?"
def _jprompt(t,a,b): return [{"role":"system","content":SYSTEM},{"role":"user","content":USER.format(t=t,a=a,b=b)}]
def _verdict(text):
    m=re.findall(r"verdict\s*:?\s*(a|b|tie)", text.lower())
    if m: return "tie" if m[-1]=="tie" else m[-1].upper()
    tail=text.lower().strip().splitlines()[-1] if text.strip() else ""
    if "tie" in tail: return "tie"
    if re.search(r"\bb\b",tail) and not re.search(r"\ba\b",tail): return "B"
    if re.search(r"\ba\b",tail) and not re.search(r"\bb\b",tail): return "A"
    return "tie"
def _load_judge():
    jid=CFG["judge"]["model"]
    tok=AutoTokenizer.from_pretrained(jid); tok.padding_side="left"
    if tok.pad_token is None: tok.pad_token=tok.eos_token
    quant=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
    m=AutoModelForCausalLM.from_pretrained(jid, quantization_config=quant, torch_dtype=torch.bfloat16, device_map="auto")
    return m.eval(), tok
@torch.no_grad()
def _jgen(m, tok, prompts, batch=4):
    outs=[]
    for i in range(0,len(prompts),batch):
        ch=prompts[i:i+batch]
        txt=[tok.apply_chat_template(x, add_generation_prompt=True, tokenize=False) for x in ch]
        enc=tok(txt, return_tensors="pt", padding=True, add_special_tokens=False).to(m.device)
        g=m.generate(**enc, max_new_tokens=256, do_sample=False, pad_token_id=tok.pad_token_id)
        for j in range(len(ch)): outs.append(tok.decode(g[j][enc["input_ids"].shape[1]:], skip_special_tokens=True))
    return outs
def judge_winrate(mt, tickets, base_ans, cand_ans):
    m, tok = mt; b=CFG["judge"].get("batch",4)
    v1=[_verdict(x) for x in _jgen(m, tok, [_jprompt(t,c,bb) for t,bb,c in zip(tickets,base_ans,cand_ans)], b)]  # cand=A
    v2=[_verdict(x) for x in _jgen(m, tok, [_jprompt(t,bb,c) for t,bb,c in zip(tickets,base_ans,cand_ans)], b)]  # cand=B
    s=0.0
    for a1,a2 in zip(v1,v2):
        c1=1.0 if a1=="A" else (0.5 if a1=="tie" else 0.0)
        c2=1.0 if a2=="B" else (0.5 if a2=="tie" else 0.0)
        s+=(c1+c2)/2.0
    return round(s/len(tickets),4) if tickets else 0.0

In [ ]:
#Gate (win-rate) + the orchestrated loop, median across seeds
def gate(base, cand, win_rate):
    a=CFG["acceptance"]
    worst=max((base[k]["score"]-cand[k]["score"]) for k in ["sentinel","reasoning","tools"])
    accept = win_rate>=a["min_win_rate"] and worst<=a["max_regression_drop"]
    print(f"[gate] win_rate={win_rate:.3f} (min {a['min_win_rate']:.3f}) | worst_drop={worst:.3f} "
          f"(max {a['max_regression_drop']:.3f}) -> {'ACCEPT' if accept else 'REJECT'}")
    return accept, win_rate, worst

def run_stage(base, base_name, data_file, lr=None, epochs=None):
    seeds=CFG["train"].get("seeds",[0]); runs=[]
    for s in seeds:
        name_s=f"{base_name}-s{s}" if len(seeds)>1 else base_name
        adapter=train_one(data_file, name_s, lr=lr, epochs=epochs, seed=s)
        res=evaluate_gen(name_s, adapter); runs.append({"seed":s,"name":name_s,"adapter":adapter,"res":res})
    mt=_load_judge()                                  # load judge once for all seeds in this stage
    try:
        for r in runs:
            wr=judge_winrate(mt, base["tickets"], base["answers"], r["res"]["answers"])
            acc,_,drop=gate(base, r["res"], wr); r.update({"win_rate":wr,"accept":acc,"drop":drop})
    finally:
        del mt; torch.cuda.empty_cache()
    return runs

def aggregate(runs):
    a=CFG["acceptance"]; wrs=[r["win_rate"] for r in runs]; drops=[r["drop"] for r in runs]
    mwr,md=statistics.median(wrs),statistics.median(drops)
    accept = mwr>=a["min_win_rate"] and md<=a["max_regression_drop"]
    passing=[r for r in runs if r["accept"]]; chosen=max(passing or runs, key=lambda r: r["win_rate"])
    print(f"[gate] across seeds: {'ACCEPT' if accept else 'REJECT'} (median win-rate={mwr:.3f}, "
          f"median drop={md:.3f}, passed={len(passing)}/{len(runs)}, range=[{min(wrs):.3f},{max(wrs):.3f}]) "
          f"-> chosen {chosen['name']}")
    return accept, chosen

base = evaluate_gen("base")
replay = CFG["guardrails"]["replay_mix"]
name1 = f"{CFG['name']}-{'replay' if replay else 'noreplay'}"
runs1 = run_stage(base, name1, "train_mix.jsonl" if replay else "train_synth.jsonl")
ok1, chosen1 = aggregate(runs1)
final = chosen1["adapter"] if ok1 else None
if not ok1:
    adj=CFG["adjust_on_fail"]; lr=CFG["train"]["lr"]*adj["lower_lr_factor"]
    print(f"\n[pipeline] gate rejected; adjusted re-run (lr={lr}, epochs={adj['reduce_epochs_to']})")
    runs2=run_stage(base, f"{CFG['name']}-adj", "train_mix.jsonl", lr=lr, epochs=adj["reduce_epochs_to"])
    ok2, chosen2=aggregate(runs2); final = chosen2["adapter"] if ok2 else None
print("\nACCEPTED:", final or "none")

In [ ]:
#Calibrate the judge against YOUR judgments (do this before trusting the gate)
# 1) writes a template pairing base vs the chosen candidate's answers for you to hand-label
import glob
def write_calibration_template(n=15):
    base=json.load(open(f"{OUT}/result_base.json"))
    cand_files=[f for f in glob.glob(f"{OUT}/result_*.json") if "result_base" not in f]
    if not cand_files: print("no candidate result yet; run the loop first"); return
    cand=json.load(open(sorted(cand_files)[-1]))
    rows=[{"ticket":t,"answer_a":b,"answer_b":c,"human":""}    # fill human with 'a' / 'b' / 'tie'
          for t,b,c in list(zip(base["tickets"], base["answers"], cand["answers"]))[:n]]
    open(f"{OUT}/calibration.jsonl","w").write("\n".join(json.dumps(r) for r in rows))
    print(f"wrote {OUT}/calibration.jsonl ({len(rows)} pairs). Fill the 'human' field with a/b/tie, then run calibrate().")

# 2) after labeling, compares the judge to your labels
def calibrate():
    rows=[json.loads(l) for l in open(f"{OUT}/calibration.jsonl") if l.strip() and json.loads(l).get("human")]
    if not rows: print("label the 'human' field in calibration.jsonl first"); return
    mt=_load_judge()
    try:
        vs=[_verdict(x) for x in _jgen(mt[0], mt[1], [_jprompt(r["ticket"], r["answer_a"], r["answer_b"]) for r in rows], CFG["judge"].get("batch",4))]
    finally:
        del mt; torch.cuda.empty_cache()
    jmap={"A":"a","B":"b","tie":"tie"}
    agree=sum(1 for r,v in zip(rows,vs) if r["human"]==jmap[v])
    dec=[(r,v) for r,v in zip(rows,vs) if r["human"] in ("a","b") and jmap[v] in ("a","b")]
    agree_dec=sum(1 for r,v in dec if r["human"]==jmap[v])
    print(f"raw agreement {agree}/{len(rows)}={agree/len(rows):.2f}" + (f" | decisive {agree_dec}/{len(dec)}={agree_dec/len(dec):.2f}" if dec else ""))
    print("rule of thumb: >= ~0.8 decisive agreement before trusting the judge to gate.")

write_calibration_template()
# after you hand-label OUT/calibration.jsonl, run:  calibrate()

In [ ]:
#Download adapters + results
import shutil, glob
made=[]
for d in sorted(glob.glob(f"{OUT}/*/")):
    if os.path.exists(os.path.join(d,"adapter_config.json")):
        base_dir=d.rstrip("/"); shutil.make_archive(base_dir,"zip",base_dir); made.append(os.path.basename(base_dir))
print("zipped adapters:", made)
print("Download the lora *.zip and result_*.json from the Output tab.")